# 📚 AI-Based Library Management System
## Module: Semantic Search + RAG Chatbot with Tool Use, Safety Guardrails & Session Memory

**Prepared by:** Amvi &nbsp;|&nbsp; Chitkara University

---

### Pipeline Overview
1. Load & preprocess the book catalog (7k Books dataset)
2. Simulate live availability data
3. Generate semantic embeddings and build a searchable vector index (FAISS)
4. Detect user emotion using a dedicated classifier model
5. Give the chatbot **tools** (function calling) to search, check availability, and recall prior recommendations — instead of relying on the LLM's memory or general knowledge
6. Enforce hard safety rules in code (not just prompts) for self-harm content
7. Maintain per-session conversation memory
8. Interactive chat loop

---

## 🔧 Bug-Fix Pass — What Was Wrong & What Changed

The chatbot was working end-to-end, but five concrete bugs were causing the symptoms you saw. Each one is called out with a `FIX:` comment at the exact line it lives on, but here's the summary:

| Symptom you saw | Root cause | Fix |
|---|---|---|
| "Top 5 books" then "suggest top from this" → a totally different book | `get_recent_recommendations` read `session["last_books"]`, a list that grew forever across the *whole* conversation — "this" silently meant "everything ever shown", not "the list I just gave you" | New `session["last_recommended"]` is **overwritten**, not appended, every round |
| Saying "ok" sometimes triggers a fresh book pitch | `tool_choice="required"` forces a tool call on *every* message; anything that slipped past the exact-match acknowledgment check had no valid "do nothing" tool to call, so the model often defaulted to `search_books` | Added a `small_talk_or_out_of_scope` tool + a more robust acknowledgment check |
| Doesn't remember what you said earlier | Conversation history was flattened into one text blob glued onto the system prompt instead of real chat turns | History is now passed as actual `user`/`assistant` messages |
| Recommends books that turn out to be unavailable | `search_books` stripped out `copies_available` / `is_available` before returning results — the model literally never received that data | Availability fields are now included in every tool result, plus an explicit rule to always disclose unavailability |
| Hallucinated answers | The `except BadRequestError` fallback explicitly told the model to *"answer directly... without using any tools"* — asking it to invent an answer from general knowledge | Fallback now returns an honest error message instead of an ungrounded guess; final-generation temperature also dropped to 0 with an extra grounding reminder |

Also added: `search_books` now excludes books already recommended this session, so "give me something else" doesn't just repeat the same 5 titles.


## 1. Setup — Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json

## 2. Load Dataset

Using the **7k Books dataset** (`Books.csv`) — chosen because it has real `description`, `subtitle`,
and `categories` text needed for semantic search, unlike the Goodreads/Book-Crossing datasets used
elsewhere in the project.

In [2]:
df = pd.read_csv('Books.csv')
print("Shape:", df.shape)
df.info()

Shape: (6810, 12)
<class 'pandas.DataFrame'>
RangeIndex: 6810 entries, 0 to 6809
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   isbn13          6810 non-null   int64  
 1   isbn10          6810 non-null   str    
 2   title           6810 non-null   str    
 3   subtitle        2381 non-null   str    
 4   authors         6738 non-null   str    
 5   categories      6711 non-null   str    
 6   thumbnail       6481 non-null   str    
 7   description     6548 non-null   str    
 8   published_year  6804 non-null   float64
 9   average_rating  6767 non-null   float64
 10  num_pages       6767 non-null   float64
 11  ratings_count   6767 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 4.3 MB


## 3. Preprocessing

### 3.1 Remove Duplicate Books

In [3]:
before = df.shape[0]
df = df.drop_duplicates(subset=['isbn13'])
print(f"Dropped {before - df.shape[0]} duplicate ISBN rows")

Dropped 0 duplicate ISBN rows


### 3.2 Drop Rows With No Description
Semantic search needs real text to embed — a book with no description can't be searched meaningfully.

In [4]:
before = df.shape[0]
df = df.dropna(subset=['description'])
print(f"Dropped {before - df.shape[0]} rows with missing description")

Dropped 262 rows with missing description


### 3.3 Fill Missing Text Fields With Meaningful Placeholders
Using `''` for a missing author causes broken output later (e.g. "by " with no name).
A real placeholder like `'Unknown Author'` avoids that.

In [5]:
df['subtitle'] = df['subtitle'].fillna('')
df['authors'] = df['authors'].fillna('Unknown Author')
df['categories'] = df['categories'].fillna('Uncategorized')
df['thumbnail'] = df['thumbnail'].fillna('https://via.placeholder.com/128x193.png?text=No+Cover')

### 3.4 Clean Text Fields (strip HTML noise / extra whitespace)

In [6]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['description'] = df['description'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)
df['categories'] = df['categories'].apply(clean_text)
df['authors'] = df['authors'].apply(clean_text)

### 3.5 Fill Missing Numeric Fields
Using the **median** (not 0) for ratings/pages — 0 would wrongly imply "zero rating" instead of "unknown".
`ratings_count` is the one exception where 0 genuinely means no ratings yet.

In [7]:
df['num_pages'] = df['num_pages'].fillna(df['num_pages'].median())
df['average_rating'] = df['average_rating'].fillna(df['average_rating'].median())
df['ratings_count'] = df['ratings_count'].fillna(0)
df['published_year'] = df['published_year'].fillna(df['published_year'].median())

### 3.6 Remove Low-Quality / Placeholder Descriptions
Some rows have junk like `"No Marketing Blurb"` instead of a real description — these hurt embedding
quality, so books with under 5 words are dropped.

In [8]:
df['description_length'] = df['description'].str.split().str.len()

very_short = df[df['description_length'] < 5]
print(f"{len(very_short)} books have descriptions under 5 words — removing them")

df = df[df['description_length'] >= 5].reset_index(drop=True)
print("Final cleaned shape:", df.shape)
print(df.isnull().sum())

56 books have descriptions under 5 words — removing them
Final cleaned shape: (6492, 13)
isbn13                0
isbn10                0
title                 0
subtitle              0
authors               0
categories            0
thumbnail             0
description           0
published_year        0
average_rating        0
num_pages             0
ratings_count         0
description_length    0
dtype: int64


## 4. Simulate Live Availability

No real checkout/inventory data exists yet, so realistic copy counts are simulated per book.
*(Documented as simulated data — a real deployment would connect to actual circulation records.)*

In [9]:
np.random.seed(42)  # reproducible results across reruns

df['total_copies'] = np.random.randint(1, 4, size=len(df))
df['copies_available'] = df['total_copies'].apply(lambda total: np.random.randint(0, total + 1))
df['is_available'] = df['copies_available'] > 0

df[['title', 'total_copies', 'copies_available', 'is_available']].head(10)

,title,total_copies,copies_available,is_available
0,Gilead,3,1,True
1,Spider's Web,1,0,False
2,The One Tree,3,3,True
3,Rage of angels,3,2,True
4,The Four Loves,1,0,False
5,The Problem of Pain,1,1,True
6,Empires of the Monsoon,3,2,True
7,The Gap Into Madness,2,2,True
8,Master of the Game,3,1,True
9,If Tomorrow Comes,3,3,True


## 5. Build the Search Text Column

In [10]:
df['search_text'] = (
    df['title'] + ' ' +
    df['subtitle'] + ' ' +
    df['categories'] + ' ' +
    df['description']
)
print(df['search_text'].iloc[0][:300])

Gilead  Fiction A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towa


## 6. Embedding Model — Sanity Check

Before embedding the full catalog, verify the model captures **meaning**, not just keywords.
Two sentences about the same theme (no shared vocabulary) should land much closer together than
two unrelated sentences.

In [11]:
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

s1 = "A young boy discovers he is a wizard and attends a magical school"
s2 = "An orphan learns he has supernatural powers and joins an academy for gifted children"
s3 = "A step-by-step guide to baking sourdough bread at home"

vecs = embed_model.encode([s1, s2, s3])

def euclidean(a, b):
    return norm(a - b)

print("s1 vs s2 (similar meaning, should be SMALL):", euclidean(vecs[0], vecs[1]))
print("s1 vs s3 (unrelated, should be LARGE):       ", euclidean(vecs[0], vecs[2]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

s1 vs s2 (similar meaning, should be SMALL): 0.94103575
s1 vs s3 (unrelated, should be LARGE):        1.4097451


## 7. Generate Embeddings for the Full Catalog + Build FAISS Index

`IndexFlatL2` performs exact search — fine at this scale (~6.5k books). At millions of vectors, an
approximate index (e.g. HNSW) would be used instead for speed.

In [12]:
embeddings = embed_model.encode(
    df['search_text'].tolist(),
    show_progress_bar=True,
    batch_size=64
)
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

Embeddings shape: (6492, 384)


In [13]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
print("Total vectors indexed:", index.ntotal)

Total vectors indexed: 6492


## 8. Semantic Search Function

In [14]:
def semantic_search(query, top_k=10):
    query_vector = embed_model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, top_k)
    results = df.iloc[indices[0]][
        ['title', 'authors', 'description', 'total_copies', 'copies_available', 'is_available']
    ].copy()
    results['distance'] = distances[0]
    return results

### Quick Test

In [15]:
print(semantic_search("magic school for young wizards")[['title', 'authors', 'distance']])

                                               title  \
907                   Mr Majeika and the School Trip   
4325             Harry Potter and the Goblet of Fire   
5563                         The Wizard's Apprentice   
2612                                    Harry Potter   
2630                     The Harry Potter Collection   
2585        The Girl, the Dragon, and the Wild Magic   
2850                        Winter of Magic's Return   
2597         Harry Potter and the Chamber of Secrets   
2598  Harry Potter and the Sorcerer's Stone (Book 1)   
3522            The lion, the witch and the wardrobe   

                          authors  distance  
907            Humphrey Carpenter  0.977796  
4325                J. K. Rowling  0.993755  
5563         Jackie French Koller  1.030673  
2612                J. K. Rowling  1.072969  
2630                J. K. Rowling  1.089233  
2585                 Dave Luckett  1.101296  
2850            Pamela F. Service  1.104136  
2597  J. K. Row

## 9. Groq API Key Setup

Loaded from a local `.env` file — never hardcoded, never committed to GitHub (`.env` is in `.gitignore`).

In [16]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Key loaded:", os.environ.get("GROQ_API_KEY") is not None)

Key loaded: True


## 10. Emotion Detection Model

A **separate, dedicated transformer** classifies the user's emotional tone. This is real model
inference — distinct from prompt engineering, which decides what to *do* with this label.

In [17]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

def detect_emotion(text):
    result = emotion_classifier(text)[0]
    top_emotion = max(result, key=lambda x: x['score'])
    return top_emotion['label'], top_emotion['score']

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Quick Test

In [18]:
print(detect_emotion("I've had such an awful, exhausting day"))
print(detect_emotion("Can you recommend a mystery novel"))

('fear', 0.6878663897514343)
('neutral', 0.6560062766075134)


## 11. Hard Safety Filter — Self-Harm / Suicide Content

**Design decision:** this is enforced in plain Python, checked *before* any LLM call — not left as a
soft, LLM-interpreted instruction. A model asked to "use judgment" about distress signals is
unreliable; a keyword-based hard block is not. If triggered, the query never reaches retrieval or
generation at all, and a crisis helpline is returned instead.

In [19]:
UNSAFE_KEYWORDS = ['suicide', 'suicidal', 'self-harm', 'self harm', 'kill myself', 'end my life']

def contains_unsafe_request(query):
    query_lower = query.lower()
    return any(keyword in query_lower for keyword in UNSAFE_KEYWORDS)

CRISIS_MESSAGE = (
    "I'm not able to help with that. If you or someone you know is struggling, please reach out to "
    "a crisis helpline — in India, you can contact AASRA at 91-22-27546669, available 24/7. "
    "Is there something else I can help you find in the library?"
)

## 12. Chatbot Tools (Function Calling)

Rather than trusting the LLM to remember prior answers or reliably decide when to search, it is
given a fixed set of **tools** — real Python functions it can call for grounded data. This is the
standard, production pattern for connecting an LLM to real data reliably (used by ChatGPT, Claude,
and most deployed AI assistants), and is more robust than either free-form prompting or hand-written
regex rules for detecting intent.

| Tool | Purpose |
|---|---|
| `search_books` | Fresh semantic search for a new topic/genre request |
| `get_book_description` | Story/summary of one specific, already-known book |
| `get_multiple_availability` | Copies available for one or more named books |
| `get_recent_recommendations` | Recall the exact books already shown in this session — used for vague follow-ups like "which of these" or "the ones you gave me," without triggering a fresh, potentially ungrounded search |


In [26]:
def get_book_description(title):
    book_row = df[df['title'] == title]
    if book_row.empty:
        return {"error": "Book not found in catalog"}
    row = book_row.iloc[0]
    return {
        "title": row['title'],
        "author": row['authors'],
        "description": row['description'],
        "copies_available": int(row['copies_available']),
        "total_copies": int(row['total_copies']),
        "is_available": bool(row['is_available'])
    }


def get_multiple_availability(titles):
    results = []
    for title in titles:
        book_row = df[df['title'] == title]
        if book_row.empty:
            results.append({"title": title, "error": "not found in catalog"})
        else:
            row = book_row.iloc[0]
            results.append({
                "title": row['title'],
                "copies_available": int(row['copies_available']),
                "total_copies": int(row['total_copies']),
                "is_available": bool(row['is_available'])
            })
    return results


def search_books(query, top_k=5, exclude_titles=None):
    exclude_titles = set(exclude_titles or [])
    raw = semantic_search(query, top_k=top_k + len(exclude_titles) + 10)
    raw = raw[~raw['title'].isin(exclude_titles)].head(top_k)
    cols = ['title', 'authors', 'description', 'total_copies', 'copies_available', 'is_available']
    return raw[cols].to_dict('records')


def get_recent_recommendations(session_id):
    session = get_session(session_id)
    titles = session["last_recommended"]
    if not titles:
        return {"error": "No books have been recommended yet in this conversation"}

    results = []
    for title in titles:
        book_row = df[df['title'] == title]
        if not book_row.empty:
            row = book_row.iloc[0]
            results.append({
                "title": row['title'],
                "author": row['authors'],
                "description": row['description'],
                "copies_available": int(row['copies_available']),
                "total_copies": int(row['total_copies']),
                "is_available": bool(row['is_available'])
            })
    return results

In [27]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_books",
            "description": "Search the library catalog for books matching a NEW topic, genre, or theme",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "What kind of book the user wants"}},
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_book_description",
            "description": "Get the plot/story/summary of a specific, already-named book",
            "parameters": {
                "type": "object",
                "properties": {"title": {"type": "string", "description": "Exact title of the book"}},
                "required": ["title"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_multiple_availability",
            "description": "Get availability for one or more specific, already-named books",
            "parameters": {
                "type": "object",
                "properties": {
                    "titles": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of exact book titles to check"
                    }
                },
                "required": ["titles"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_recent_recommendations",
            "description": (
                "Get the exact books already recommended MOST RECENTLY in this conversation, with "
                "full details. Use this -- NOT search_books -- whenever the user refers back to the "
                "last list shown using words like 'these', 'this', 'those', 'the ones you gave me', "
                "'top from this', 'top suggestion from that list', 'which one', 'from what you showed me'."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "small_talk_or_out_of_scope",
            "description": (
                "Use this for greetings, thanks, goodbyes, small talk, or anything that is NOT a "
                "book search, a book-detail question, an availability check, or a reference back to "
                "earlier recommendations."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]


## 13. Session-Based Conversation Memory

Memory is keyed **per session**, not stored in one shared global list — the correct architecture for
a system with multiple concurrent users (this dashboard will have many students using it at once).
Each session tracks the last few conversation turns and the actual titles already recommended, so
follow-up questions can be resolved reliably instead of relying on the LLM to "remember" on its own.

*(For a production deployment, this would be persisted to a lightweight database, e.g. SQLite, so
memory survives an app restart — not required for this demo's scope.)*

In [28]:


sessions = {}  # {session_id: {"history": [...], "last_recommended": [...], "all_recommended_titles": {...}}}

def get_session(session_id):
    if session_id not in sessions:
        sessions[session_id] = {
            "history": [],
            "last_recommended": [],
            "all_recommended_titles": set()
        }
    return sessions[session_id]

## 14. Small Deterministic Helpers

Some decisions are more reliable when made in plain code rather than left to the LLM's judgment:
- Recognizing a plain acknowledgment ("ok", "thanks") so the bot doesn't push more recommendations
- Deciding whether to open with a brief empathetic line, based on the *actual* emotion label and a
  real confidence threshold — not the LLM guessing at tone from vague instructions

In [29]:
import re as _re

ACKNOWLEDGMENT_PHRASES = ['ok', 'okay', 'okk', 'k', 'got it', 'thanks', 'thank you', 'thanks a lot',
                           'cool', 'nice', 'alright', 'sounds good', 'perfect', 'great']

def is_pure_acknowledgment(query):
    cleaned = _re.sub(r'[^\w\s]', '', query.lower()).strip()
    return cleaned in ACKNOWLEDGMENT_PHRASES

NEGATIVE_EMOTIONS = ['sadness', 'anger', 'fear']

def should_acknowledge_mood(emotion, confidence):
    return emotion in NEGATIVE_EMOTIONS and confidence > 0.6


## 15. RAG Chatbot — Full Pipeline

Order of checks for every incoming message:
1. **Hard safety filter** (Section 11) — blocks self-harm content before anything else runs
2. **Pure acknowledgment check** — skips the LLM entirely for "ok"/"thanks"
3. **Tool-calling LLM call** — `tool_choice="required"` forces the model to always call a real tool
   rather than ever answering from its own general knowledge, guaranteeing grounding
4. **Tool execution** — the actual Python function runs, returning real catalog data
5. **Final response generation** — the LLM turns the tool's raw result into a natural answer
6. **Session update** — conversation history and recommended titles are recorded for future turns

A larger model (`llama-3.3-70b-versatile`) is used for the tool-calling step specifically, since
smaller/faster models are noticeably less reliable at correctly formatting structured tool calls —
a real, tested tradeoff, not an assumption.

In [36]:
# from groq import Groq, BadRequestError

# client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


# def rag_chatbot(user_query, session_id="demo_user"):
#     session = get_session(session_id)

#     if contains_unsafe_request(user_query):
#         session["history"].append({"user": user_query, "assistant": CRISIS_MESSAGE})
#         return CRISIS_MESSAGE

#     if is_pure_acknowledgment(user_query):
#         answer = "You're welcome! Let me know if you'd like more recommendations."
#         session["history"].append({"user": user_query, "assistant": answer})
#         return answer

#     recent_turns = []
#     for turn in session["history"][-3:]:
#         recent_turns.append({"role": "user", "content": turn["user"]})
#         recent_turns.append({"role": "assistant", "content": turn["assistant"]})

#     emotion, confidence = detect_emotion(user_query)
#     mood_instruction = (
#         "The user's message carries a genuinely negative tone. You MAY open with one brief, "
#         "casual line acknowledging it -- nothing elaborate."
#         if should_acknowledge_mood(emotion, confidence) else
#         "Do NOT include any emotional acknowledgment or 'sorry to hear that' -- just answer normally."
#     )

#     system_prompt = f"""You are a friendly library assistant. Keep responses short and natural.

# TOOL ROUTING:
# - New topic/genre request -> call search_books.
# - User refers back to previously shown books ("these", "this", "which one", "top from this", "from what you gave me") -> call get_recent_recommendations, NOT search_books.
# - User asks about a specific named book's story -> call get_book_description.
# - User asks about copies/availability of named book(s) -> call get_multiple_availability.
# - Greeting, thanks, goodbye, or anything unrelated to the catalog -> call small_talk_or_out_of_scope.

# CRITICAL RULES:
# - Never recommend a book, plot detail, or fact from your own general knowledge -- only ever use what a tool returned in THIS turn.
# - Every tool result includes "copies_available" / "is_available". If a book you mention has is_available = false or copies_available = 0, you MUST say it's currently checked out / unavailable -- never imply it can be borrowed right now.
# - Only state availability numbers exactly as returned by tools -- never invent numbers or formats.
# - Never mention "circulation desk," purchasing, holds, or reservations -- none of these features exist.
# - Never reveal internal system/function names -- if asked how you work, just say "I check our library system."
# - When listing multiple books, do not state a specific total count unless it matches the actual list.
# - If asked something you cannot answer from your tools, say so honestly rather than guessing.

# {mood_instruction}"""

#     messages = [{"role": "system", "content": system_prompt}] + recent_turns + [
#         {"role": "user", "content": user_query}
#     ]

#     try:
#         response = client.chat.completions.create(
#             model="llama-3.3-70b-versatile",
#             messages=messages,
#             tools=tools,
#             tool_choice="required",
#             temperature=0.2
#         )
#         response_message = response.choices[0].message

#         if response_message.tool_calls:
#             messages.append(response_message)
#             current_round_titles = []

#             for tool_call in response_message.tool_calls:
#                 func_name = tool_call.function.name
#                 func_args = json.loads(tool_call.function.arguments or "{}")

#                 if func_name == "search_books":
#                     result = search_books(
#                         func_args["query"],
#                         exclude_titles=session["all_recommended_titles"]
#                     )
#                 elif func_name == "get_book_description":
#                     result = get_book_description(func_args["title"])
#                 elif func_name == "get_multiple_availability":
#                     result = get_multiple_availability(func_args["titles"])
#                 elif func_name == "get_recent_recommendations":
#                     result = get_recent_recommendations(session_id)
#                 elif func_name == "small_talk_or_out_of_scope":
#                     result = {"note": "No catalog lookup needed -- just respond naturally and briefly, and offer to help find a book."}
#                 else:
#                     result = {"error": "unknown function"}

#                 if isinstance(result, list):
#                     for item in result:
#                         title = item.get("title")
#                         if title:
#                             current_round_titles.append(title)

#                 messages.append({
#                     "role": "tool",
#                     "tool_call_id": tool_call.id,
#                     "content": json.dumps(result)
#                 })

#             if current_round_titles:
#                 session["last_recommended"] = current_round_titles
#                 session["all_recommended_titles"].update(current_round_titles)

#             messages.append({
#                 "role": "system",
#                 "content": (
#                     "Reminder: only mention books, facts, and numbers that appear in the tool "
#                     "results above. Do not add any book not present there. If is_available is "
#                     "false for a book, say clearly that it's currently unavailable/checked out."
#                 )
#             })

#             final_response = client.chat.completions.create(
#                 model="llama-3.1-8b-instant",
#                 messages=messages,
#                 temperature=0
#             )
#             answer = final_response.choices[0].message.content
#         else:
#             answer = response_message.content

#         if "<function=" in answer or "<function =" in answer:
#             answer = "I'm having trouble checking that right now -- could you ask about one book at a time?"

#     except BadRequestError:
#         answer = "Sorry, I had trouble processing that -- could you rephrase your question?"

#     session["history"].append({"user": user_query, "assistant": answer})
#     return answer

# ------------1st CHATBOTT PURA KMM KRR REHAAA








# UPDATED YAAAAAAAAAAAAA
from groq import Groq, BadRequestError

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


def _call_llm(messages, session, session_id, retry_hint=None):
    """Runs the full tool-call -> tool-execute -> final-generation flow once.
    Raises BadRequestError up to the caller if the tool-call step fails to format."""
    msgs = list(messages)
    if retry_hint:
        msgs.append({"role": "system", "content": retry_hint})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=msgs,
        tools=tools,
        tool_choice="required",
        temperature=0.2
    )
    response_message = response.choices[0].message

    if not response_message.tool_calls:
        return response_message.content, []

    msgs.append(response_message)
    current_round_titles = []

    for tool_call in response_message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments or "{}")

        if func_name == "search_books":
            # exclude titles already recommended earlier in the session AND
            # titles already picked up earlier IN THIS SAME TURN (e.g. two
            # search_books calls back to back to build a "top 10" list)
            exclude = session["all_recommended_titles"].union(current_round_titles)
            result = search_books(func_args["query"], exclude_titles=exclude)
        elif func_name == "get_book_description":
            result = get_book_description(func_args["title"])
        elif func_name == "get_multiple_availability":
            result = get_multiple_availability(func_args["titles"])
        elif func_name == "get_recent_recommendations":
            result = get_recent_recommendations(session_id)
        elif func_name == "small_talk_or_out_of_scope":
            result = {"note": "No catalog lookup needed -- just respond naturally and briefly, and offer to help find a book."}
        else:
            result = {"error": "unknown function"}

        if isinstance(result, list):
            for item in result:
                title = item.get("title")
                if title:
                    current_round_titles.append(title)

        msgs.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    msgs.append({
        "role": "system",
        "content": (
            "Reminder: only mention books, facts, and numbers that appear in the tool "
            "results above. Do not add any book not present there. If is_available is "
            "false for a book, say clearly that it's currently unavailable/checked out. "
            "If a tool result contains an 'error' key (book not found / no summary / "
            "no recommendations yet), say so plainly and honestly -- do NOT fill the gap "
            "with a plot summary, fact, or detail from your own knowledge, even if you "
            "happen to know the real book. Never say stalling phrases like 'let me check' "
            "or 'let me look into that' -- the tool has ALREADY run; state the actual "
            "result directly."
        )
    })

    final_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=msgs,
        temperature=0
    )
    return final_response.choices[0].message.content, current_round_titles


def rag_chatbot(user_query, session_id="demo_user"):
    session = get_session(session_id)

    if contains_unsafe_request(user_query):
        session["history"].append({"user": user_query, "assistant": CRISIS_MESSAGE})
        return CRISIS_MESSAGE

    if is_pure_acknowledgment(user_query):
        answer = "You're welcome! Let me know if you'd like more recommendations."
        session["history"].append({"user": user_query, "assistant": answer})
        return answer

    recent_turns = []
    for turn in session["history"][-3:]:
        recent_turns.append({"role": "user", "content": turn["user"]})
        recent_turns.append({"role": "assistant", "content": turn["assistant"]})

    emotion, confidence = detect_emotion(user_query)
    mood_instruction = (
        "The user's message carries a genuinely negative tone. You MAY open with one brief, "
        "casual line acknowledging it -- nothing elaborate."
        if should_acknowledge_mood(emotion, confidence) else
        "Do NOT include any emotional acknowledgment or 'sorry to hear that' -- just answer normally."
    )

    last_book_hint = ""
    if len(session["last_recommended"]) == 1:
        last_book_hint = f"\nThe single book most recently discussed is \"{session['last_recommended'][0]}\" -- if the user's message is a vague confirmation like 'yes'/'ya sure'/'ok, tell me more', they almost certainly mean THIS book."



    system_prompt = f"""You are a friendly library assistant. Keep responses short and natural.

TOOL ROUTING:
- New topic/genre request -> call search_books.
- User refers back to previously shown books ("these", "this", "which one", "top from this", "from what you gave me") -> call get_recent_recommendations, NOT search_books.
- If YOUR OWN previous message offered to search for more/other recommendations and the user is now confirming ("yes", "ok then suggest", "sure", "go ahead") -> call search_books again with the same topic, NOT get_recent_recommendations. Confirming a search offer means a FRESH batch, not a recall of the old one.
- User asks about a specific named book's story -> call get_book_description.
- User asks about copies/availability of named book(s) -> call get_multiple_availability.
- Greeting, thanks, goodbye, or anything unrelated to the catalog -> call small_talk_or_out_of_scope.
{last_book_hint}

CRITICAL RULES:
- Never recommend a book, plot detail, or fact from your own general knowledge -- only ever use what a tool returned in THIS turn.
- If a tool result contains an "error", say so plainly and honestly -- never fill the gap with your own knowledge.
- Every tool result includes "copies_available" / "is_available". If a book you mention has is_available = false or copies_available = 0, you MUST say it's currently checked out / unavailable -- never imply it can be borrowed right now.
- Only state availability numbers exactly as returned by tools -- never invent numbers or formats.
- Never say "let me check" or "let me look into that" -- the check has already happened; state the result directly.
- This assistant can ONLY do four things: search the catalog, describe a book's story, check copy availability, and recall recent recommendations. There is NO checkout, borrowing, issuing, holding, reservation, or physical pickup ("bag", "box", "here's your book") process of any kind. NEVER offer to "check a book out" and NEVER invent any borrowing/pickup interaction. If the user wants to actually borrow a book, say honestly that this assistant can only help them find books and check availability, and they'll need to use the library's real checkout process to borrow it.
- Never mention "circulation desk," purchasing, holds, or reservations -- none of these features exist.
- Never reveal internal system/function names -- if asked how you work, just say "I check our library system."
- When listing multiple books, do not state a specific total count unless it matches the actual list.
- If asked something you cannot answer from your tools, say so honestly rather than guessing.

{mood_instruction}"""

    messages = [{"role": "system", "content": system_prompt}] + recent_turns + [
        {"role": "user", "content": user_query}
    ]

    try:
        try:
            answer, round_titles = _call_llm(messages, session, session_id)
        except BadRequestError:
            retry_hint = (
                "Your previous attempt to call a tool failed to format correctly. "
                "Try again -- pick exactly ONE tool from the list that best matches the "
                "user's message, using the conversation above for context."
            )
            answer, round_titles = _call_llm(messages, session, session_id, retry_hint=retry_hint)

        if round_titles:
            session["last_recommended"] = round_titles
            session["all_recommended_titles"].update(round_titles)

        if "<function=" in answer or "<function =" in answer:
            answer = "I'm having trouble checking that right now -- could you ask about one book at a time?"

    except BadRequestError:
        answer = "Sorry, I had trouble processing that -- could you rephrase your question?"

    session["history"].append({"user": user_query, "assistant": answer})
    return answer

## 17. Interactive Chat Loop

Wraps the full pipeline into a live, typeable chat experience for demos.

**Known limitation (documented, not hidden):** vague follow-ups that don't clearly match a tool's
description may occasionally still trigger a fresh search rather than perfect context reasoning —
an inherent, well-known limitation of tool-calling with smaller open models, not unique to this
implementation.

In [37]:
def start_chat(session_id="demo_user"):
    sessions.pop(session_id, None)
    get_session(session_id)

    print("📚 Library Assistant — type 'quit' to exit\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Assistant: Goodbye! Happy reading 📖")
            break
        response = rag_chatbot(user_input, session_id=session_id)
        print(f"\nAssistant: {response}\n")

start_chat()

📚 Library Assistant — type 'quit' to exit



You:  suggest me some fantasy books



Assistant: Here are some fantasy book recommendations:

1. "The Tough Guide to Fantasyland" by Diana Wynne Jones - 1 copy available
2. "Legends" by Robert Silverberg - 1 copy available
3. "The Tough Guide to Fantasyland" by Diana Wynne Jones - 2 copies available
4. "Legends" by George R. R. Martin and Anne McCaffrey - currently unavailable
5. "The Book of the Dragon" - currently unavailable

Would you like more recommendations?



You:  which one from these would you recommend



Assistant: I don't have a personal preference, but if you're interested in learning more about the fantasy genre, "The Tough Guide to Fantasyland" by Diana Wynne Jones might be a good choice. Would you like to know more about it?



...


Assistant: I'm just a library assistant.



You:  which api ur using



Assistant: I'm not using any specific API. I'm just a library assistant helping you find books in our catalog.



You:  quit


Assistant: Goodbye! Happy reading 📖
